# How to find late filers: NT 10-K and NT 10-Q notices

An NT 10-K or NT 10-Q means a company can't file its report on time. Find them by date, ticker and form, and check whether the report followed.

Read it with charts at https://www.quantisnow.com/sec-api/guides/late-filings-nt-10-k-nt-10-q. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A company that will miss a 10-K or 10-Q deadline files Form 12b-25, which EDGAR lists as `NT 10-K` or `NT 10-Q`. It buys 15 extra days for an annual report and 5 for a quarterly one. Most late filers are small and file inside the extension, but a notice that cites an audit still in progress or a restatement is one of the more reliable early warnings in the filing record.

### Which companies filed a late-filing notice for their 10-Q in the last 30 days?

An NT 10-Q means the quarterly report will be late. Short sellers and credit desks screen for it.

```shell
curl -X POST https://www.quantisnow.com/api/v1/filings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:\"NT 10-Q\" AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 100):

```json
{
  "total": {
    "value": 100,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001829126-26-012616",
      "accessionNo": "0001829126-26-012616",
      "formType": "NT 10-Q",
      "filedAt": "2026-11-18T16:05:02-05:00",
      "cik": "0002083452",
      "companyName": "Shreya Acquisition Group",
      "ticker": "SAGU",
      "tickers": [
        "SAGU"
      ],
      "items": [],
      "description": "SEC Form NT 10-Q filed by Shreya Acquisition Group",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/2083452/000182912626012616/0001829126-26-012616-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-nt-10-q-filed-by-shreya-acquisition-group-6798685"
    },
    {
      "id": "0001493152-26-052886",
      "accessionNo": "0001493152-26-052886",
      "formType": "NT 10-Q",
      "filedAt": "2026-11-17T17:00:11-05:00",
      "cik": "0001853070",
      "companyName": "SOAR AI Inc.",
      "ticker": "SOAR",
      "tickers": [
        "SOAR"
      ],
      "items": [],
      "description": "SEC Form NT 10-Q filed by SOAR AI Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1853070/000149315226052886/0001493152-26-052886-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-nt-10-q-filed-by-soar-ai-inc-6797234"
    }
  ]
}
```

## Did the report follow?

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()

notices = search("filings", 'formType:"NT 10-K" AND filedAt:[2026-03-01 TO 2026-04-15]', size=50)["data"]
for n in notices:
    after = search("filings", f'cik:{n["cik"]} AND formType:"10-K" AND filedAt:>={n["filedAt"][:10]}', size=1)
    filed = after["data"][0]["filedAt"][:10] if after["data"] else "not yet"
    print(n["companyName"], n["filedAt"][:10], "->", filed)

## Why they were late

The reason is in the notice's text. [Full-text search](https://www.quantisnow.com/sec-api/full-text-search) with `formTypes: ["NT 10-K", "NT 10-Q"]` and a phrase like `"restatement"` or `"material weakness"` finds the ones that matter.

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

const res = await fetch(BASE + "/full-text", {
  method: "POST",
  headers: { Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY, "Content-Type": "application/json" },
  body: JSON.stringify({ query: '"restatement"', formTypes: ["NT 10-K", "NT 10-Q"], sort: "filedAt:desc", size: 20 }),
});
for (const f of (await res.json()).data) console.log(f.companyName, f.highlights[0]);
```